# sam3dbody → GLB — walkthrough

Convert a `.sam3dbody` file (SAM 3D Body inference output) into an animated GLB.

**Before you start:**
1. `conda create -n sam3d python=3.12 -y && conda activate sam3d`
2. `pip install -r ../requirements.txt`
3. Download the MHR assets into `../assets/` and extract `mhr_head_buffers.npz` — see [`../assets/README.md`](../assets/README.md).
4. Run this notebook with the `sam3d` kernel.

## 0. Setup paths

On Linux, `pymomentum` needs torch's shared libs on `LD_LIBRARY_PATH`. Setting it from inside the process *before* importing pymomentum usually works; if you hit a segfault, set it in the shell before launching Jupyter instead.

In [ ]:
import os, sys, subprocess, pathlib

REPO = pathlib.Path.cwd().parent          # repo root (this notebook lives in notebooks/)
INPUT_DIR  = REPO / "input"
OUTPUT_DIR = REPO / "output"
ASSETS_DIR = REPO / "assets"
EXPORT_SCRIPT = REPO / "export_glb_pymomentum.py"

import torch
torch_lib = os.path.join(os.path.dirname(torch.__file__), "lib")
os.environ["LD_LIBRARY_PATH"] = torch_lib + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

print("repo:   ", REPO)
print("assets: ", ASSETS_DIR, "exists:", ASSETS_DIR.is_dir() and any(ASSETS_DIR.glob('lod*.fbx')))
print("buffers:", (ASSETS_DIR / 'mhr_head_buffers.npz').exists())

## 1. Pick a `.sam3dbody` file

Drop one (or more) into the `input/` folder. We'll use the first one found.

In [ ]:
sam_files = sorted(INPUT_DIR.glob("*.sam3dbody"))
assert sam_files, f"Put a .sam3dbody file in {INPUT_DIR}"
src = sam_files[0]
print("Using:", src.name)

## 2. Peek inside it (optional)

In [ ]:
import numpy as np
d = np.load(src, allow_pickle=True)
print("Keys:", list(d.keys()))
for k in d:
    a = d[k]
    print(f"  {k}: shape={getattr(a,'shape',None)} dtype={getattr(a,'dtype','')}")
if "metadata" in d:
    print("metadata:", d["metadata"].item() if d["metadata"].shape == () else d["metadata"][0])

## 3. Convert to GLB

We shell out to `export_glb_pymomentum.py` (pymomentum + a long-running notebook kernel can be fragile in the same process). Tweak `OPTS` as you like:

| flag | effect |
|------|--------|
| `--lod N` | mesh detail, 0 = 73K verts … 6 = lowest |
| `--freeze-legs` | hold legs in rest pose |
| `--freeze-root` | hold global rotation fixed |
| `--smooth S` | gaussian temporal smoothing (frames) |
| `--anim-only` | animation only, no mesh (~0.8 MB) |
| `--every N` | keep every Nth frame |

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
dst = OUTPUT_DIR / (src.stem + ".glb")
OPTS = ["--freeze-legs", "--freeze-root", "--smooth", "2"]   # good defaults for talking-head / signing clips

cmd = [sys.executable, str(EXPORT_SCRIPT), str(src), "-o", str(dst), *OPTS]
print(" ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr); raise SystemExit("export failed")
print(f"\u2713 {dst}  ({dst.stat().st_size/1e6:.1f} MB)")

## 4. Convert everything in `input/` at once

Same as running `python batch_convert.py` from the repo root.

In [ ]:
res = subprocess.run([sys.executable, str(REPO / "batch_convert.py")], text=True)
print("return code:", res.returncode)
print("output files:", sorted(p.name for p in OUTPUT_DIR.glob("*.glb")))

## 5. View the GLB

- Drag the file from `output/` onto <https://gltf-viewer.donmccurdy.com/>
- Or in Blender: *File ▸ Import ▸ glTF 2.0*
- Or load it with Three.js / Babylon.js `GLTFLoader`

For web apps that need a static mesh + streamed animation clips, see
`tools/build_static_mesh_glb.py` and the `--anim-only` flag (README § "Web apps").